<a href="https://colab.research.google.com/github/yuprotsyk/bigdata-course/blob/main/notebooks/topic03_dataframe_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Аналіз та обробка великих даних

Ю.С. Процик. Курс лекцій

# Тема 3. DataFrame API та Spark SQL: Ефективна обробка даних в Apache Spark

### План

1. [Нова парадигма обробки даних: Чому Spark SQL?](#1.-Нова-парадигма-обробки-даних:-Чому-Spark-SQL?)
2. [Основні абстракції: DataFrame та Dataset](#2.-Основні-абстракції:-DataFrame-та-Dataset)
3. [Архітектура та механізми оптимізації](#3.-Архітектура-та-механізми-оптимізації)
4. [Створення DataFrame та інтерфейси I/O](#4.-Створення-DataFrame-та-інтерфейси-I\/O)
5. [Управління схемами](#5.-Управління-схемами)
6. [Типи даних DataFrame](#6.-Типи-даних-DataFrame)
7. [Програмна модель обробки](#7.-Програмна-модель-обробки)
8. [SQL-інтерфейс та метадані](#8.-SQL-інтерфейс-та-метадані)
9. [Базові ETL-операції з DataFrame API](#9.-Базові-ETL-\-операції-з-DataFrame-API)
10. [Групування та агрегування даних](#10.-Групування-та-агрегування-даних)
11. [Реляційні операції](#11.-Реляційні-операції)
12. [Особливості роботи зі складними типами даних](#12.-Особливості-роботи-зі-складними-типами-даних)
13. [Корисні ресурси](#13.-Корисні-ресурси)

## 1. Нова парадигма обробки даних: Чому Spark SQL?

Згідно з [офіційною документацією](https://spark.apache.org/docs/latest/sql-programming-guide.html), **Spark SQL** – це спеціалізований модуль Apache Spark призначений для опрацювання структурованих даних. На відміну від базового API RDD, інтерфейси Spark SQL надають системі детальну інформацію про структуру даних та характер обчислень, що виконуються.

**Ключові особливості нової парадигми:**

- **Вирішення проблеми "Black Box":** у RDD Spark сприймає дані як набір непрозорих об'єктів. Spark SQL, навпаки, "бачить" схему даних (метадані), що дозволяє системі використовувати цю додаткову інформацію для проведення внутрішніх оптимізацій.

- **Уніфікований механізм виконання (Unified Execution Engine):** незалежно від того, яку мову (Python, Scala, Java, R) або API (SQL чи Dataset API) ви використовуєте, для обчислення результату застосовується той самий рушій виконання. Це означає, що розробники можуть вільно перемикатися між мовами та інтерфейсами без втрати продуктивності.

## 2. Основні абстракції: DataFrame та Dataset

У Spark SQL існують дві ключові абстракції, які дозволяють працювати зі структурованими даними набагато ефективніше, ніж у базовому RDD API.

### DataFrame: Таблиця у розподіленому середовищі

**DataFrame** – це розподілена колекція записів, організована в іменовані стовпці. Концептуально це еквівалент таблиці в реляційній базі даних або фрейму даних у мовах R чи Python, але з набагато потужнішими внутрішніми оптимізаціями.

**Основні характеристики DataFrame:**

- **Схема (Schema):** Кожен DataFrame має чітку структуру, яка описує назви та типи даних усіх його стовпців.

- **Джерела даних:** DataFrames можна будувати з широкого спектра джерел: структурованих файлів (JSON, CSV, Parquet, ORC), таблиць у Hive, зовнішніх баз даних або існуючих RDD.

- **SQL-сумісність:** Вони мають вбудовану підтримку багатьох стандартних функцій SQL та реляційних операторів, зокрема з’єднань (JOINs).

- **Відмовостійкість:** DataFrames оцінюються як спрямовані ациклічні графи (DAG), що забезпечує можливість відновлення даних у разі збоїв та дозволяє відстежувати історію походження даних (lineage).

### Dataset: Поєднання типізації та швидкості

**Dataset** – це інтерфейс (доданий у Spark 1.6), який поєднує переваги RDD (сувора типізація, можливість використання лямбда-функцій) із перевагами оптимізованого механізму виконання Spark SQL.

**Особливості Dataset API:**

- **Мовна підтримка:** Цей API доступний лише для мов Scala та Java.

- **Type-safety:** Помилки в типах даних будуть знайдені вже на етапі компіляції (у Scala/Java), що робить код надійнішим.

- **Зв'язок із DataFrame:** У Scala DataFrame – це просто псевдонім типу (type alias) для `Dataset[Row]`. У Java користувачі використовують `Dataset<Row>` для представлення DataFrame.

- **Python специфіка:** У PySpark немає Dataset API. Однак завдяки динамічній природі Python, багато переваг цього інтерфейсу вже доступні автоматично (наприклад, природний доступ до полів рядка за ім'ям: `row.columnName`).


## 3. Архітектура та механізми оптимізації

На відміну від RDD, де розробник сам відповідає за ефективність коду, Spark SQL використовує додаткову інформацію про структуру даних та обчислень для автоматичного підвищення продуктивності. Оптимізація відбувається на трьох рівнях: логічному плануванні, фізичному виконанні та динамічному коригуванні під час роботи.

### Catalyst Optimizer (Оптимізація логіки)

**Catalyst Optimizer** – це інтелектуальний механізм, який перетворює операції над DataFrame в оптимізований план виконання. Він застосовує два типи оптимізацій: **Rule-based** (на основі правил) та **Cost-based** (на основі вартості).

Незалежно від мови програмування (Python, SQL чи Scala), Catalyst проводить запит через такі фази:

1. **Analysis (Аналіз):** перевірка імен стовпців і таблиць за допомогою каталогу метаданих. *Unresolved Logical Plan* стає *Analyzed Logical Plan*.

2. **Logical Optimization (Логічна оптимізація):** спрощення запиту без зміни результату. Тут діють стратегії **Predicate Pushdown** (фільтрація даних у джерелі) та **Constant Folding** (обчислення констант на кшталт 24 * 60 один раз для всіх рядків).

3. **Physical Optimizations (Фізична оптимізація):** Spark генерує кілька фізичних планів і за допомогою моделі вартості обирає найефективніший (наприклад, обирає між *Broadcast Join* та *SortMerge Join*).

4. **Code Generation (Генерація коду):** фінальний етап, де план перетворюється на байт-код Java для виконання.

![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0301.png)

### Project Tungsten (Оптимізація пам'яті та CPU)

**Tungsten** – це рушій фізичного виконання, який використовує **стовпцевий формат (columnar format)** у пам'яті для всіх DataFrames.

- **Binary Memory Management:** Tungsten зберігає дані в бінарному форматі поза "купою" Java (**off-heap memory management**). Це усуває накладні витрати на об'єкти Java та запобігає тривалим паузам Garbage Collector.

- **Whole-Stage Code Generation:** Spark генерує єдиний Java-цикл для цілого ланцюжка операцій, що дозволяє CPU працювати максимально ефективно.

- **Photon Engine (специфічно для Databricks):** Векторизований рушій, який обробляє дані пакетами (batches), що значно прискорює обчислення порівняно з порядковою обробкою.

**Adaptive Query Execution (AQE)** – це механізм динамічного коригування планів під час виконання. На основі фактичних статистичних показників, отриманих у процесі виконання запиту, Spark може змінювати стратегію виконання з метою підвищення продуктивності.

### Стовпцеве зберігання даних (Columnar Storage)

![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0302-databricks.png)

Окрім бінарного управління пам'яттю, Spark SQL використовує переваги стовпцевої моделі даних для аналітичних запитів.

- **Організація даних за стовпцями:** На відміну від традиційних баз даних, які зберігають дані рядками (Row Oriented), стовпцеве зберігання групує дані одного типу разом. Це дозволяє зчитувати лише ті стовпці, які необхідні для конкретного запиту, значно зменшуючи обсяг операцій введення-виведення (I/O).

- **Ефективність для аналітики:** Такий підхід є особливо ефективним для аналітичних навантажень (OLAP), де часто потрібно обчислити агрегати (суму, середнє) по одному стовпцю серед мільярдів записів.

- **Реалізація в екосистемі:**

  - **Внутрішнє зберігання:** використовується в оперативній пам'яті при кешуванні DataFrame.

  - **Фізичні формати:** реалізовано в популярних форматах файлів, таких як **Parquet** та **ORC**, що дозволяє Spark виконувати "проштовхування стовпців" (Column Pruning) безпосередньо на рівні диска.

### Інструмент розробника: Аналіз плану через `.explain()`

Для розуміння внутрішніх процесів Spark використаємо метод `.explain()`. Це вікно в роботу системи, яке дозволяє проаналізувати фінальну стратегію виконання ще до фактичного старту обчислень.

- `.explain()` **(базовий режим):** виводить лише Physical Plan (фізичний план).

- `.explain(True)` **(розширений режим):** відображає всю еволюцію запиту – від початкового коду до оптимізованого результату. Саме цей режим дозволяє наочно побачити "магію" Catalyst Optimizer.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Lec_Architecture") \
    .getOrCreate()

# Створюємо DataFrame
data = [(1, "Ivan", 25), (2, "Maria", 19), (3, "Oleg", 30)]
df = spark.createDataFrame(data, ["id", "name", "age"])

# Приклад запиту
result = df.select("name", "age").filter(F.col("age") > 21)

# Виклик детального плану
result.explain(True)

== Parsed Logical Plan ==
'Filter '`>`('age, 21)
+- Project [name#4, age#5L]
   +- LogicalRDD [id#3L, name#4, age#5L], false

== Analyzed Logical Plan ==
name: string, age: bigint
Filter (age#5L > cast(21 as bigint))
+- Project [name#4, age#5L]
   +- LogicalRDD [id#3L, name#4, age#5L], false

== Optimized Logical Plan ==
Project [name#4, age#5L]
+- Filter (isnotnull(age#5L) AND (age#5L > 21))
   +- LogicalRDD [id#3L, name#4, age#5L], false

== Physical Plan ==
*(1) Project [name#4, age#5L]
+- *(1) Filter (isnotnull(age#5L) AND (age#5L > 21))
   +- *(1) Scan ExistingRDD[id#3L,name#4,age#5L]



**Як читати план виконання:**

- **Блоки тексту (зверху вниз):** Це еволюція плану – від вашого коду до фінальної інструкції Spark.

- **Логіка всередині блоку (знизу вгору):** Це шлях даних. Читаємо від джерела (Scan) в самому низу до фінального результату вгорі.

**1. Parsed Logical Plan (синтаксичний аналіз)**

Це пряма трансляція вашого коду в дерево операцій.

- **Що бачимо:** `Filter` стоїть над `Project`. Spark просто зафіксував черговість ваших команд, ще не перевіряючи, чи існують такі стовпці.

**2. Analyzed Logical Plan (перевірка метаданих)**

На цьому етапі Spark звертається до Catalog.

- **Що бачимо:** З'являються типи даних (наприклад, `age: bigint`). Spark підтвердив, що стовпці існують. Також бачимо `cast(21 as bigint)` – автоматичне приведення типів для коректного порівняння.

**3. Optimized Logical Plan (робота Catalyst)**

Тут відбувається "магія" оптимізації.

- **Що змінилося:** `Filter` і `Project` помінялися місцями.

- **Predicate Pushdown:** Spark вирішив спочатку виконати фільтрацію, щоб не витрачати ресурси на формування стовпців (`Project`) для рядків, які все одно будуть видалені.

- **Додаткові умови:** Додався фільтр `isnotnull(age)`, щоб заздалегідь відсіяти порожні значення.

**4. Physical Plan (генерація коду Tungsten)**

Це фінальна інструкція для процесорів кластера.

- **Whole-Stage Code Generation:** Символ `*(1)` означає, що Spark злив `Filter` та `Project` в один компактний Java-цикл.

- **Ефективність:** Замість того, щоб передавати дані від однієї операції до іншої, Spark обробляє рядок за один прохід у бінарному форматі.

## 4. Створення DataFrame та інтерфейси I/O

![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0303-databricks.png)

DataFrames можуть бути створені з широкого спектра джерел:

- **Файлові системи:**

  - *структуровані та напівструктуровані:* Parquet, ORC, JSON, CSV.

  - *неструктуровані:* текстові та бінарні файли.

- **Сховища таблиць (Table Formats):** Delta Lake, Apache Iceberg або Apache Hudi.

- **Системні каталоги:** таблиці та представлення (views) у Hive або Unity Catalog.

- **Зовнішні бази даних:** підключення через JDBC/ODBC (наприклад, PostgreSQL, MySQL).

- **Об’єкти в пам'яті:** перетворення з існуючих RDD або локальних колекцій Python (наприклад, списків чи Pandas DataFrames).

![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0304-databricks.png)


Для взаємодії із зовнішніми джерелами даних Spark надає два уніфіковані програмні інтерфейси, які складають основу системи введення-виведення (I/O).

- **DataFrameReader** (`spark.read`) використовується для завантаження даних у Spark. Він підтримує роботу з різноманітними форматами, такими як JSON, CSV, Parquet, ORC, а також зовнішніми базами даних та існуючими RDD. Ключовою особливістю є можливість автоматичного **визначення схеми (schema inference)**, що дозволяє Spark самостійно ідентифікувати типи даних під час зчитування.

- **DataFrameWriter** (`dataframe.write`) використовується для збереження оброблених даних у зовнішні сховища. Він забезпечує гнучкість у виборі форматів виводу, підтримує партиціонування та пропонує різні режими збереження, зокрема:
  - **overwrite** – повний перезапис існуючих даних;
  - **append** – додавання нових записів до вже існуючих.

**Основна перевага цих інтерфейсів** полягає в ідентичності синтаксису: для розробника не має значення, чи дані зчитуються з локального диска, чи записуються у хмарне сховище – програмний код залишається незмінним, забезпечуючи надійну та масштабовану обробку.

## 5. Управління схемами

![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0305-databricks.png)

Кожен DataFrame має визначену схему, яка описує структуру та типи даних його стовпців. Схема може бути визначена автоматично на основі даних (**inferred**) або задана явно для більшої ефективності. Формати, що мають вбудовані метадані, такі як **Parquet**, автоматично надають інформацію про схему, що позбавляє необхідності її ручного опису або вгадування. Для перевірки поточної структури DataFrame використовується метод `printSchema()`.

Наведений вище фрагмент коду демонструє явне визначення схеми за допомогою `StructType` та `StructField` у PySpark.

![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0306-databricks.png)

Альтернативою `StructType` є використання DDL-рядків (**Data Definition Language**), які забезпечують більш лаконічний та читабельний формат. Цей підхід особливо зручний для специфікації схем під час зчитування текстових даних (наприклад, CSV чи JSON).

Наведений вище фрагмент коду демонструє визначення схеми за допомогою DDL-рядка та його застосування під час створення DataFrame.

### Чому явна схема краща за "вгадування"?

- **Швидкість:** Spark не витрачає час на сканування всього файлу, щоб зрозуміти типи.

- **Надійність:** Якщо в стовпці з числами раптом з'явиться текст, Spark обробить це згідно з вашою схемою (наприклад, запише `null)`, а не змінить тип усього стовпця.

- **Економія пам'яті:** Ви можете чітко вказати `ByteType` або `ShortType` замість стандартного `LongType`.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Описуємо схему вручну
user_schema = StructType([
    StructField("id", IntegerType(), False),      # (Назва, Тип, чи може бути Null)
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

# Прив'язуємо схему до даних
data = [(1, "Ivan", 23), (2, "Maria", 19), (3, "Oleg", 25)]
df = spark.createDataFrame(data, schema=user_schema)

df.printSchema()

root
 |-- id: integer (nullable = false)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)



## 6. Типи даних DataFrame

### Примітивні типи даних (Primitive Data Types)

Кожен DataFrame у Spark має чітко визначену схему, яка описує структуру та типи даних усіх стовпців. Розуміння типів даних є критично важливим для забезпечення цілісності даних (**schema enforcement** – примусове дотримання схеми) та оптимізації виконання запитів.

Багато стандартних типів даних, що використовуються в програмуванні та SQL, представлені у Spark SQL як спеціальні класи Python API (`pyspark.sql.types.DataType`). Ці типи мають прямі еквіваленти в мові SQL (для використання в DDL-схемах) та нативні відповідники в мові Python.

| pyspark.sql.types.DataType | Тип у Spark SQL | Базовий еквівалент у Python |
|----------------------------|---------------|------------------------|
| `ByteType`                 | `TINYINT`     | `int`                  |
| `ShortType`                | `SMALLINT`    | `int`                  |
| `IntegerType`              | `INT`         | `int`                  |
| `LongType`                 | `BIGINT`      | `long`                 |
| `FloatType`                | `FLOAT`       | `float`                |
| `DoubleType`               | `DOUBLE`      | `float`                |
| `BooleanType`              | `BOOLEAN`     | `bool`                 |
| `StringType`               | `STRING`      | `str`                  |
| `BinaryType`               | `BINARY`      | `bytearray`            |
| `TimestampType`            | `TIMESTAMP`   | `datetime.datetime`    |
| `DateType`                 | `DATE`        | `datetime.date`        |


### Складні типи даних (Complex Data Types)

Окрім примітивних типів, Spark SQL надає підтримку для складних типів даних, які дозволяють описувати вкладені та ієрархічні структури прямо в стовпцях DataFrame. Це критично важливо при роботі з напівструктурованими даними (наприклад, JSON-файлами) або складними аналітичними форматами типу Parquet.

| pyspark.sql.types.DataType | Тип у Spark SQL | Базовий еквівалент у Python      | Особливості                                      |
|---------------|---------|----------------------------------|-----------------------------------------------|
| `ArrayType`   | `ARRAY` | `list` (або `tuple`)             | Впорядкована колекція елементів **одного типу**            |
| `MapType`     | `MAP`   | `dict`                           | Набір пар "ключ-значення". <br>Ключі та значення можуть мати **різні типи**     |
| `StructType`  | `STRUCT`| `tuple`, `dict`, `namedtuple`    | Іменовані поля з заданими типами. <br>Фактично це **таблиця з наперед визначеними стовпцями**, <br>вкладена в одну комірку            |

Детальніше див. в [офіційній документації PySpark щодо типів даних](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/data_types.html)



- **`ArrayType` (масиви):** використовується, коли в одному стовпці потрібно зберігати список значень, наприклад, список ідентифікаторів замовлень користувача або набір тегів до статті.

- **`MapType` (словники):** ідеально підходить для зберігання гнучких наборів атрибутів, де назви ключів можуть змінюватися (наприклад, технічні характеристики товару: колір, вага, розмір).

- **`StructType` (структури):** фундамент для створення ієрархічних моделей даних. За допомогою `StructType` можна згрупувати логічно пов'язані поля (наприклад, об'єкт `address`, що містить поля `city`, `street` та `zip_code`) в один стовпець.

## 7. Програмна модель обробки

### Трансформації та Дії

DataFrames є **незмінними (immutable)** – після створення їхні дані не можуть бути модифіковані. Замість зміни існуючого об'єкта, будь-яка операція повертає новий DataFrame, що дозволяє Spark відстежувати історію походження даних (**lineage**) та забезпечувати відмовостійкість.

Всі операції в Spark API поділяються на дві категорії:

- **Трансформації (Transformations)** створюють нові DataFrames на основі існуючих (наприклад, `select()`, `filter()`, `groupBy()`). Вони є лінивими операціями – це означає, що вони лише будують логічний план обробки, не запускаючи реальні обчислення негайно.

- **Дії (Actions)** (наприклад, `show()`, `count()`, `write()`) ініціюють фактичне виконання обчислень, створюючи так званий **Spark Job**. Тільки в цей момент Spark обробляє дані та повертає результат драйверу або записує його у зовнішнє сховище.

Модель **лінивих обчислень (lazy evaluation)** є фундаментальною для Spark SQL, оскільки вона дозволяє системі оптимізувати продуктивність. Замість того, щоб виконувати кожну команду крок за кроком, Spark чекає на виклик дії, аналізує весь ланцюжок трансформацій і формує єдиний оптимізований план виконання, який мінімізує використання ресурсів і час обробки.

### Основні методи DataFrame API

**Трансформації**

| Метод | Опис |
|-------|------|
| `select()` | Вибір конкретних стовпців з DataFrame |
| `filter()` | Фільтрація рядків за умовою (псевдонім `where()`) |
| `withColumn()` | Додавання або зміна стовпця в DataFrame |
| `groupBy()` | Групування рядків DataFrame за вказаними стовпцями |
| `agg()` | Агрегація рядків після групування |



**Дії**

| Метод | Опис |
|-------|------|
| `count()` | Повертає кількість рядків у DataFrame |
| `show()` | Відображає вміст DataFrame |
| `take(n)` | Повертає перші `n` рядків |
| `first()` | Повертає перший рядок |
| `write()` | Зберігає DataFrame у сховище |

Детальніше див. в [офіційній документації PySpark DataFrame API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html)


## 8. SQL-інтерфейс та метадані

Розглянемо можливості використання стандартного синтаксису SQL для маніпуляцій з DataFrames та механізми управління метаданими.

### SparkSQL та реєстрація представлень

Spark дозволяє виконувати SQL-запити безпосередньо над DataFrames. Для цього DataFrame необхідно зареєструвати як представлення (view):

- **Тимчасові представлення (`createOrReplaceTempView`):** існують лише в межах поточної Spark-сесії.

- **Глобальні представлення (`createGlobalTempView`):** доступні в усіх сесіях одного Spark-застосунку.

Після реєстрації можна використовувати команду `spark.sql()` для виконання будь-яких стандартних SQL-операцій, що забезпечує безшовну інтеграцію між SQL та DataFrame API

In [ ]:
# DataFrame з попереднього прикладу
df.show()

# Реєструємо таблицю в пам'яті Spark
df.createOrReplaceTempView("users")

# Виконуємо запит
sql_results = spark.sql("""
    SELECT name, age
    FROM users
    WHERE age > 21
    ORDER BY age DESC
""")

sql_results.show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1| Ivan| 23|
|  2|Maria| 19|
|  3| Oleg| 25|
+---+-----+---+

+----+---+
|name|age|
+----+---+
|Oleg| 25|
|Ivan| 23|
+----+---+



### SQL Metastores

- **Metastore:** відповідає за зберігання метаданих – описів схем таблиць, їхнього фізичного розташування на диску, партицій та визначень функцій.

- **Unity Catalog (специфічно для Databricks):** це централізована служба метаданих, яка забезпечує деталізовану безпеку та управління активами даних, включаючи контроль доступу та відстеження походження даних (lineage) на рівні всього робочого простору.

## 9. Базові ETL-операції з DataFrame API

**ETL (Extract, Transform, Load)** – це концепція організації процесу обробки даних, яка передбачає їх вилучення з джерел, перетворення відповідно до вимог бізнес-логіки та подальше завантаження до цільових систем зберігання або аналітики.

Процес ETL у Spark базується на використанні DataFrame API як уніфікованого інтерфейсу. Завдяки тому, що Spark SQL використовує один і той самий рушій виконання незалежно від мови (Python, Scala, SQL), розробники можуть вільно перемикатися між програмними методами та SQL-запитами.



### Методи трансформації та їх зіставлення з SQL

Трансформації – це основні інструменти для розподіленого перетворення даних. Вони автоматично розподіляються між партиціями кластера, що дозволяє паралельно обробляти великі масиви даних. Кожна операція повертає новий DataFrame, підтримуючи принцип незмінності (immutability).

Трансформації DataFrame мають еквівалентні SQL-операції, що дозволяє використовувати знайому реляційну парадигму.

| Метод DataFrame API     | Еквівалент SQL | Функція                                                         |
|--------------------------|---------------|------------------------------------------------------------------|
| `select()`               | `SELECT`        | Вибір конкретних стовпців                      |
| `filter()`, `where()`    | `WHERE`         | Фільтрація рядків за заданою умовою                            |
| `groupBy()`              | `GROUP BY`      | Групування рядків (зазвичай передує агрегації)                 |
| `orderBy()`, `sort()`    | `ORDER BY`      | Сортування даних у DataFrame за вказаними стовпцями           |
| `join()`                 | `JOIN`          | Об'єднання двох DataFrames за спільним ключем                  |

Детальніше див. в [офіційній документації PySpark DataFrame API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html)

### Обробка відсутніх значень (Missing Values)

![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0307-databricks.png)


**Відсутні значення (missing values)** – це поширена проблема, з якою зазвичай доводиться стикатися під час організації ETL-процесів або відображення джерел на цільові структури. У реальних задачах дані часто містять пропущені або `null` значення, які необхідно врахувати, зафіксувати або замінити. Це має вирішальне значення для підтримання якості даних та забезпечення точного аналізу.

- Для перевірки наявності пропущених значень використовуються методи `isNull()` та `isNotNull()`.

- Якщо виконати `count()` із зазначенням конкретного стовпця, буде підраховано лише ненульові значення – тобто рядки, де цей стовпець не містить `null`.

- Щоб замінити `null` на певне значення (наприклад, `0`, порожній рядок `""`, `"not applicable"` або `"not supplied"`), можна скористатися методами DataFrame `fillna()` або `na.fill()`.

- Якщо потрібно видалити рядки з пропущеними значеннями, застосовуються `dropna()` або `na.drop()`.


### Способи доступу до стовпців DataFrame

![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0308-databricks.png)

У Spark існує кілька способів **доступу до стовпців DataFrame**.

- Найпростіший варіант – прямий вибір через `df.select("name")`.

- Можливий також доступ через атрибут об'єкта – `df.column_name`, але цей спосіб працює лише для назв, що є коректними ідентифікаторами Python (без пробілів і спеціальних символів).

- Універсальним способом є звернення через квадратні дужки – `df["column name"]`, що дозволяє працювати з будь-якими назвами стовпців.

- **Найкращою практикою для складних трансформацій** вважається використання об'єкта колонки (`col`), оскільки це відкриває доступ до специфічних методів обробки:
  - `alias()`: перейменування стовпця для уникнення конфліктів.
  - `cast()` / `astype()`: зміна типу даних (наприклад, з `DoubleType` на `IntegerType`).
  - `asc()` / `desc()`: визначення напрямку сортування при впорядкуванні даних.



![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0309-databricks.png)

### Вбудовані функції Spark (Built-in Functions)

Spark надає широкий набір вбудованих функцій для обробки та аналізу DataFrame.  
Вони працюють як зі скалярними типами (рядки, числа, логічні значення), так і зі складними (масиви, структури).

| Категорія функцій | Призначення | Приклади функцій |
|------------------|------------|-----------------|
| **Math**         | Математичні обчислення | `abs()`, `ceil()`, `floor()`, `round()`, `sqrt()` |
| **Datetime**     | Робота з датами та часом | `current_date()`, `current_timestamp()`, `date_add()`, `year()` |
| **Collection**   | Операції з масивами та колекціями | `array()`, `explode()`, `size()` |
| **Bitwise**      | Побітові операції | `bitwise_and()`, `bitwise_or()`, `shiftLeft()`, `shiftRight()` |
| **Aggregate**    | Агрегатні функції | `sum()`, `avg()`, `count()`, `max()`, `min()` |
| **Window**       | Віконні функції для аналітики | `row_number()`, `rank()`, `dense_rank()`, `lead()`, `lag()` |

Важливою перевагою є те, що ці функції можна використовувати як у DataFrame API, так і безпосередньо в SQL-запитах, оскільки вони мають ідентичні назви в обох інтерфейсах.

#### Деякі поширені вбудовані функції

| Вбудована функція       | SQL-еквівалент | Опис |
|-------------------------|----------------|------------------------------------------------|
| `round(col, scale)`     | `ROUND`          | Округлення числа до заданої точності |
| `concat(col1, col2)`    | `CONCAT`        | Об’єднання рядків (конкатенація) |
| `date_format(col, fmt)` | `DATE_FORMAT`    | Форматування дати у рядок за заданим шаблоном |
| `regexp_replace(col, pattern, replace)` | `REGEXP_REPLACE` | Заміна частин рядка за допомогою регулярного виразу |
| `coalesce(col1, col2)`  | `COALESCE`       | Повертає перше ненульове значення |

Для детальнішої інформації див.:  
- [Документація PySpark Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)  
- [Документація Spark SQL](https://spark.apache.org/docs/latest/api/sql/index.html)

### Функції користувача (UDFs)

![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0310-databricks.png)

**UDF (User Defined Functions, функції користувача)** розширюють можливості Spark, дозволяючи застосовувати власні функції Python до стовпців DataFrame. Вони дають змогу реалізовувати складну логіку обробки даних, яку неможливо або складно реалізувати за допомогою вбудованих функцій, а також забезпечують повторне використання коду через інкапсуляцію типових перетворень.

Водночас UDF характеризуються нижчою продуктивністю, оскільки **не оптимізуються засобами Catalyst Optimizer** та спричиняють додаткові накладні витрати, пов’язані із серіалізацією даних між середовищем Python і JVM.

Згідно з рекомендаціями щодо продуктивності, доцільно використовувати вбудовані функції всюди, де це можливо, і застосовувати UDF лише у випадках, коли відсутня альтернативна реалізація стандартними засобами Spark.

![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0311-databricks.png)

**Pandas UDF (Pandas User Defined Functions)** – це високопродуктивна альтернатива стандартним Python UDF. Вони використовують Apache Arrow для ефективного обміну даними між JVM Spark та середовищем Python без зайвого копіювання.

Векторизоване виконання дозволяє обробляти дані пакетами (batches), що мінімізує накладні витрати на серіалізацію та десеріалізацію. Висока швидкість досягається завдяки використанню стовпцевого формату Arrow в оперативній пам'яті.

Проте, завжди варто надавати пріоритет вбудованим функціям Spark, оскільки вони оптимізуються на рівні рушія виконання. Pandas UDF слід обирати лише для специфічних трансформацій або складних агрегацій, які важко реалізувати нативними засобами.

### Приклад побудови ETL-конвеєра (PySpark & Parquet)

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

# 1. Ініціалізація
spark = SparkSession.builder.appName("ProductionETL").getOrCreate()

# 2. EXTRACT: Сувора схема (Data Contract)
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("raw_title", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("specs", MapType(StringType(), StringType()), True), # Технічні деталі
    StructField("tags", ArrayType(StringType()), True)              # Теги та категорії
])

# Життєві дані: пропуски, зайві пробіли, NULL у вкладених структурах
raw_data = [
    (1, "  iPhone 15 Pro  ", 1200.0, {"color": "Titanium", "storage": "256GB"}, ["mobile", "apple", "sale"]),
    (2, "Samsung S23 ", 850.0, {"color": "Black", "camera": "50MP"}, ["mobile", "android"]),
    (3, "MacBook Air M2", None, {"ram": "16GB", "cpu": "M2"}, ["laptop", "apple"]),
    (4, " Generic Tablet ", -50.0, None, []), # Помилкова ціна та порожні структури
    (5, None, 300.0, {"type": "Refurbished"}, ["clearance"])
]

df = spark.createDataFrame(raw_data, schema)

# 3. TRANSFORM: Векторизована обробка (Pandas UDF)
@F.pandas_udf(StringType())
def clean_title(titles: pd.Series) -> pd.Series:
    # Очищення: видалення пробілів, перетворення у верхній регістр, обробка NULL
    return titles.fillna("UNKNOWN PRODUCT").str.strip().str.upper()

# Основна трансформація
df_transformed = df.select(
    F.col("id"),
    clean_title(F.col("raw_title")).alias("product_name"),

    # Бізнес-логіка: виправляємо ціну (якщо < 0 або NULL -> ставимо 0.0)
    F.when(F.col("price") > 0, F.col("price")).otherwise(0.0).alias("final_price"),

    # Робота з Map: дістаємо колір, якщо він є
    F.coalesce(F.col("specs").getItem("color"), F.lit("Standard")).alias("color_option"),

    # Робота з Array: розгортаємо категорії для аналітики
    F.explode_outer(F.col("tags")).alias("category")
)

# 4. LOAD: Збереження в Parquet
output_path = "products_catalog.parquet"
df_transformed.write.mode("overwrite").parquet(output_path)

#

# 5. ПЕРЕВІРКА: Читання з фільтрацією (демонстрація Columnar Storage)
check_df = spark.read.parquet(output_path)
check_df.filter(F.col("category") == "apple").show()

spark.stop()

+---+--------------+-----------+------------+--------+
| id|  product_name|final_price|color_option|category|
+---+--------------+-----------+------------+--------+
|  1| IPHONE 15 PRO|     1200.0|    Titanium|   apple|
|  3|MACBOOK AIR M2|        0.0|    Standard|   apple|
+---+--------------+-----------+------------+--------+



## 10. Групування та агрегування даних

### Основи операцій `groupBy`

- Операції  `groupBy` у Spark перерозподіляють дані між вузлами кластера на основі стовпців групування.

- Агрегації виконуються паралельно у межах партицій для досягнення максимальної продуктивності.

- Підхід аналогічний синтаксису `GROUP BY` у SQL, але надає гнучкіші можливості для налаштування.

- Підтримується групування за декількома стовпцями для реалізації складних аналітичних сценаріїв.

- Метод `groupBy()` повертає об'єкт `GroupedData`, що дозволяє будувати ланцюжки агрегацій (`count`, `sum`, `avg`). Обчислення виконуються ліниво (lazy evaluation) і запускаються лише після виклику дії (action).

![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0312-databricks.png)

### Базові методи агрегування

Для обчислення статистичних показників у межах кожної групи використовуються стандартні методи, що мають прямі еквіваленти в SQL.

| Метод                 | SQL-еквівалент   | Опис                                      |
|-----------------------|-----------------|------------------------------------------|
| `count()`             | `COUNT(*)`        | Підрахунок кількості рядків у кожній групі         |
| `sum(col)`            | `SUM(col)`        | Сума значень стовпця для кожної групи   |
| `avg(col)`            | `AVG(col)`        | Середнє значення стовпця для кожної групи |
| `min(col)` / `max(col)` | `MIN / MAX`     | Мінімальне / максимальне значення стовпця у групі |

Детальніше див. в [документації Spark SQL - Grouping](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/grouping.html)


### Комбінування кількох агрегацій (`agg`)

![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0313-databricks.png)

Коли функціональності базових методів (таких як `.count()`, `.sum()` чи `.avg()`) недостатньо, використовується універсальний метод `.agg()`. Він призначений для виконання складних агрегацій над об'єктом `GroupedData` і дозволяє:

- **Комбінувати кілька різних обчислень** над однією групою в межах одного запиту (наприклад, одночасно розраховувати суму та середнє значення для різних стовпців).

- **Використовувати словниковий синтаксис** для зручного зіставлення стовпців із відповідними агрегатними функціями: `.agg({"salary": "sum", "age": "avg"})`.

- **Надавати псевдоніми (aliases)** результуючим стовпцям за допомогою методу `.alias()`. Це є критично важливим для забезпечення читабельності коду та уникнення конфліктів назв при подальшій роботі з DataFrame.

### Віконні функції (Window Functions)

**Агрегування без втрати деталізації на рівні рядків**

#### Що це таке?

Це спеціальні функції, які виконують обчислення над набором рядків,
пов’язаних із поточним рядком, зберігаючи при цьому кожен рядок у результаті.

#### Коли використовуються?

- Обчислення кумулятивних підсумків або ковзних середніх значень (наприклад, сума з накопиченням).
- Ранжування записів у межах груп (наприклад, визначення топ-позицій).
- Отримання значень попереднього або наступного рядка (через функції `lag()` та `lead()`).
- Порівняння значень окремого рядка з агрегованими показниками групи (наприклад, наскільки зарплата працівника відрізняється від середньої по відділу).

**Ключова відмінність:** На відміну від агрегацій `groupBy`, які згортають рядки в підсумкову статистику, віконні функції зберігають кожен окремий запис, додаючи до нього результати обчислень, що дозволяє бачити одночасно і детальні дані, і їхній зв'язок із групою.

![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0314-databricks.png)

Розглянемо покроково приклад використання віконних функцій.

1. **Підготовка та імпорт необхідних інструментів:** Першим кроком є імпорт спеціального класу `Window` із модуля `pyspark.sql.window` та всіх необхідних вбудованих функцій SQL. Це базове налаштування, яке відкриває доступ до аналітичних можливостей Spark.

2. **Створення специфікації вікна (Window Specification):** Далі необхідно визначити логіку, за якою буде сформоване "вікно" обробки. У даному прикладі ми створюємо об'єкт `windowSpeс`, який:
- Розподіляє дані на групи за стовпцем `"department"` (`partitionBy("department")`).
- Впорядковує записи всередині кожної групи за рівнем заробітної плати (`orderBy("salary")`).

3. **Розрахунок та додавання нових стовпців:** За допомогою методу `withColumn()` ми додаємо до DataFrame нові аналітичні показники для кожного рядка. Це дозволяє застосовувати функції безпосередньо до визначеного вікна:
- `rank()`: визначає ранг (позицію) кожного працівника всередині його відділу на основі зарплати.
- `sum()`: обчислює кумулятивні (накопичувальні) суми у межах відділу.
- `lag()`: отримує значення попереднього рядка в межах групи з урахуванням сортування.

### Аналітичний приклад: агрегація та віконні функції

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DecimalType

spark = SparkSession.builder.appName("AnalyticsFinalCorrected").getOrCreate()

# 1. СТВОРЕННЯ СХЕМИ ДЛЯ СИРИХ ДАНИХ
raw_schema = StructType([
    StructField("Department", StringType(), True),
    StructField("Employee", StringType(), True),
    StructField("Sales", DoubleType(), True)
])

data = [
    ("Electronics", "Ivan", 12000.33333333),
    ("Electronics", "Anna", 15000.55555555),
    ("Electronics", "Oleg", 10000.0),
    ("Home & Garden", "Maria", 8000.44444444),
    ("Home & Garden", "Petro", 18000.99999999),
    ("Home & Garden", "Olena", 12000.11111111),
    ("Books", "Dmytro", 5000.0)
]

# Створюємо початковий DataFrame
df_raw = spark.createDataFrame(data, raw_schema)

# 2. ОЧИЩЕННЯ: Приведення до Decimal(15, 2)
# Це прибирає "хвости" і готує дані до фінансової аналітики
df = df_raw.withColumn("Sales", F.col("Sales").cast(DecimalType(15, 2)))

# --- БЛОК 1: АГРЕГАЦІЯ (groupBy) ---
dept_stats = df.groupBy("Department").agg(
    F.count("Employee").alias("Staff_Count"),
    F.sum("Sales").alias("Total_Dept_Sales"),
    # Повторний cast після avg, щоб уникнути нових хвостів після ділення
    F.avg("Sales").cast(DecimalType(15, 2)).alias("Avg_Dept_Sales")
)

print("Результат агрегації (groupBy):")
dept_stats.show()

# --- БЛОК 2: ВІКОННІ ФУНКЦІЇ (Window) ---
# Специфікація вікна: групуємо за департаментом, сортуємо за продажами
window_spec = Window.partitionBy("Department").orderBy(F.col("Sales").desc())

analytical_df = df.withColumn(
    "Rank_In_Dept", F.rank().over(window_spec)
).withColumn(
    "Total_Dept_Sales", F.sum("Sales").over(window_spec)
).withColumn(
    # Обчислюємо частку у % від загальних продажів відділу
    "Sales_Percentage", F.round((F.col("Sales") / F.col("Total_Dept_Sales")) * 100, 2)
)

print("Аналітичний звіт з віконними функціями:")
analytical_df.show()

spark.stop()

Результат агрегації (groupBy):
+-------------+-----------+----------------+--------------+
|   Department|Staff_Count|Total_Dept_Sales|Avg_Dept_Sales|
+-------------+-----------+----------------+--------------+
|  Electronics|          3|        37000.89|      12333.63|
|Home & Garden|          3|        38001.55|      12667.18|
|        Books|          1|         5000.00|       5000.00|
+-------------+-----------+----------------+--------------+

Аналітичний звіт з віконними функціями:
+-------------+--------+--------+------------+----------------+----------------+
|   Department|Employee|   Sales|Rank_In_Dept|Total_Dept_Sales|Sales_Percentage|
+-------------+--------+--------+------------+----------------+----------------+
|        Books|  Dmytro| 5000.00|           1|         5000.00|          100.00|
|  Electronics|    Anna|15000.56|           1|        15000.56|          100.00|
|  Electronics|    Ivan|12000.33|           2|        27000.89|           44.44|
|  Electronics|    Ole

## 11. Реляційні операції

Реляційні операції у Spark базуються на принципах реляційної алгебри. Це дозволяє інтегрувати розподілені дані з різних джерел у єдину структуру (DataFrame), використовуючи спільні ключі або визначені логічні зв’язки.

### Операції з’єднання (Join)

Метод `.join()` дозволяє об’єднувати два DataFrame на основі збігу значень у ключових стовпцях. Виділяють наступні типи з’єднань:

- **Inner Join (внутрішнє з'єднання):** формує результуючий DataFrame лише з тих рядків, для яких знайдено відповідність ключів в обох наборах даних. Це стратегія за замовчуванням.

- **Left (Outer) Join (ліве зовнішнє з'єднання):** зберігає всі записи лівого DataFrame. Якщо в правому DataFrame відповідність відсутня, атрибути правої сторони заповнюються значеннями`null`.

- **Right (Outer) Join (праве зовнішнє з'єднання):** дзеркальна операція – зберігає всі записи правого DataFrame. Якщо в лівому DataFrame відповідність відсутня, атрибути лівої сторони заповнюються значеннями `null`.

- **Full Outer Join (повне зовнішнє з'єднання):** реалізує повне об'єднання, зберігаючи всі записи з обох DataFrame. У разі відсутності парних ключів з будь-якого боку, відсутні атрибути заміщуються значеннями `null`.

- **Cross Join (декартовий добуток):** виконує повне комбінаторне об'єднання кожного рядка першої таблиці з кожним рядком другої. Вимагає обережності у використанні через ризик різкого зростання обсягу даних.

**Вирішення конфліктів імен**

Якщо обидва DataFrame мають стовпці з однаковими іменами (наприклад, `id`), Spark видасть помилку неоднозначності. Для її вирішення рекомендується:

- **Використовувати псевдоніми (Aliases):** присвоїти DataFrame короткі імена перед з'єднанням.

- **Пряме звернення:** звертатися до стовпців через об'єкт конкретної таблиці (наприклад, `df1.id == df2.id`).

- **Перейменування:** змінити назву конфліктного стовпця за допомогою `.withColumnRenamed()` перед початком операції.


![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0316-databricks.png)

### Приклад на різні типи Join

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("JoinExamples").getOrCreate()

# 1. Створення даних
# Клієнти: Ivan (ID 1), Anna (ID 2), Oleg (ID 3 - без замовлень)
clients_data = [(1, "Ivan"), (2, "Anna"), (3, "Oleg")]
clients_schema = ["id", "name"]

# Замовлення: два від Івана (ID 1), одне від Анни (ID 2), одне від гостя (ID 99)
orders_data = [(101, 1, "Smartphone"), (102, 1, "Case"), (103, 2, "Laptop"), (104, 99, "UnknownItem")]
orders_schema = ["order_id", "client_id", "product"]

df_clients = spark.createDataFrame(clients_data, clients_schema)
df_orders = spark.createDataFrame(orders_data, orders_schema)

print("--- Таблиця Клієнтів ---")
df_clients.show()
print("--- Таблиця Замовлень ---")
df_orders.show()

# ==========================================
# ТИПИ З'ЄДНАНЬ (JOINS)
# ==========================================

# 1. INNER JOIN (Тільки ті, хто зробив замовлення)
print("1. INNER JOIN: Тільки клієнти з замовленнями")
df_clients.join(df_orders, df_clients.id == df_orders.client_id, "inner").show()

# 2. LEFT JOIN (Всі клієнти, навіть без замовлень)
# Олег отримає null у стовпцях замовлення
print("2. LEFT JOIN: Всі клієнти (Олег має null)")
df_clients.join(df_orders, df_clients.id == df_orders.client_id, "left").show()

# 3. FULL OUTER JOIN (Всі клієнти + всі замовлення)
print("3. FULL OUTER JOIN: Всі дані з обох сторін")
df_clients.join(df_orders, df_clients.id == df_orders.client_id, "full").show()

# ==========================================
# ВИРІШЕННЯ КОНФЛІКТІВ ІМЕН (Ambiguity)
# ==========================================
# Якщо в обох DF є стовпець 'id', використовуємо Aliases
df_c = df_clients.alias("c")
df_o = df_orders.withColumnRenamed("client_id", "id").alias("o") # навмисно створюємо конфлікт назв 'id'

print("Вирішення конфлікту імен через Alias:")
df_c.join(df_o, F.col("c.id") == F.col("o.id"), "inner") \
    .select("c.id", "c.name", "o.product") \
    .show()

spark.stop()

--- Таблиця Клієнтів ---
+---+----+
| id|name|
+---+----+
|  1|Ivan|
|  2|Anna|
|  3|Oleg|
+---+----+

--- Таблиця Замовлень ---
+--------+---------+-----------+
|order_id|client_id|    product|
+--------+---------+-----------+
|     101|        1| Smartphone|
|     102|        1|       Case|
|     103|        2|     Laptop|
|     104|       99|UnknownItem|
+--------+---------+-----------+

1. INNER JOIN: Тільки клієнти з замовленнями
+---+----+--------+---------+----------+
| id|name|order_id|client_id|   product|
+---+----+--------+---------+----------+
|  1|Ivan|     101|        1|Smartphone|
|  1|Ivan|     102|        1|      Case|
|  2|Anna|     103|        2|    Laptop|
+---+----+--------+---------+----------+

2. LEFT JOIN: Всі клієнти (Олег має null)
+---+----+--------+---------+----------+
| id|name|order_id|client_id|   product|
+---+----+--------+---------+----------+
|  1|Ivan|     102|        1|      Case|
|  1|Ivan|     101|        1|Smartphone|
|  3|Oleg|    NULL|     NU

### Операції над множинами (Set Operations)

Операції над множинами базуються на теорії множин і призначені для вертикального об'єднання або порівняння даних. Вони маніпулюють кількістю рядків, а не стовпців, як це відбувається у випадку з JOIN.

Для виконання операцій **DataFrame повинні мати сумісні схеми**: однакову кількість стовпців та відповідність типів даних за позиціями.

| Метод             | Опис                                                  | Поведінка щодо дублікатів                     |
|------------------|-------------------------------------------------------|----------------------------------------------|
| `.union()`        | Об'єднує рядки за позицією стовпців                  | Зберігає дублікати (аналог SQL `UNION ALL`) |
| `.unionByName()`  | Об'єднує рядки, зіставляючи імена стовпців           | Зберігає дублікати                           |
| `.intersect()`    | Повертає рядки, що присутні в обох DataFrame         | Автоматично видаляє дублікати                |
| `.subtract()`     | Повертає рядки першого DataFrame, відсутні в другому | Автоматично видаляє дублікати                |

### Приклад на операції над множинами

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Ініціалізація сесії
spark = SparkSession.builder \
    .appName("SetOperationsExample") \
    .getOrCreate()

# 2. Створення тестових даних (з навмисними дублікатами)
# Зверніть увагу: ("Anna", 30) є в обох списках
data1 = [("Ivan", 25), ("Anna", 30), ("Ivan", 25)] # Тут є дублікат Івана всередині
data2 = [("Anna", 30), ("Oleg", 35), ("Dmytro", 40)]

schema = ["Name", "Age"]

df1 = spark.createDataFrame(data1, schema)
df2 = spark.createDataFrame(data2, schema)

print("--- Вхідні DataFrame ---")
print("DF1 (День 1):")
df1.show()
print("DF2 (День 2):")
df2.show()

# ==========================================
# ОПЕРАЦІЇ НАД МНОЖИНАМИ
# ==========================================

# 1. UNION (Зберігає ВСІ дублікати – як внутрішні, так і між таблицями)
print("--- 1. UNION (зберігає дублікати) ---")
union_all_df = df1.union(df2)
union_all_df.show()
# Результат: 6 рядків (два Івана, дві Анни)

# 2. UNION + DISTINCT (Класичне об'єднання множин – без дублікатів)
print("--- 2. UNION + DISTINCT (унікальні записи) ---")
union_distinct_df = df1.union(df2).distinct()
union_distinct_df.show()
# Результат: 4 рядки (тільки унікальні люди)

# 3. INTERSECT (Тільки спільні записи, дублікати видаляються автоматично)
print("--- 3. INTERSECT (спільні записи) ---")
intersect_df = df1.intersect(df2)
intersect_df.show()
# Результат: Тільки ("Anna", 30)

# 4. SUBTRACT (Різниця: що є в DF1, чого немає в DF2)
print("--- 4. SUBTRACT (різниця DF1 - DF2) ---")
subtract_df = df1.subtract(df2)
subtract_df.show()
# Результат: Тільки Ivan та Oleg (якщо він був у першій).
# У нашому випадку: ("Ivan", 25), причому лише одна копія.

# 5. UNION BY NAME (Якщо стовпці в іншому порядку)
df2_reversed = df2.select("Age", "Name")
print("--- 5. UNION BY NAME (безпечне об'єднання за назвами) ---")
df1.unionByName(df2_reversed).show()

spark.stop()

--- Вхідні DataFrame ---
DF1 (День 1):
+----+---+
|Name|Age|
+----+---+
|Ivan| 25|
|Anna| 30|
|Ivan| 25|
+----+---+

DF2 (День 2):
+------+---+
|  Name|Age|
+------+---+
|  Anna| 30|
|  Oleg| 35|
|Dmytro| 40|
+------+---+

--- 1. UNION (зберігає дублікати) ---
+------+---+
|  Name|Age|
+------+---+
|  Ivan| 25|
|  Anna| 30|
|  Ivan| 25|
|  Anna| 30|
|  Oleg| 35|
|Dmytro| 40|
+------+---+

--- 2. UNION + DISTINCT (унікальні записи) ---
+------+---+
|  Name|Age|
+------+---+
|  Ivan| 25|
|  Anna| 30|
|Dmytro| 40|
|  Oleg| 35|
+------+---+

--- 3. INTERSECT (спільні записи) ---
+----+---+
|Name|Age|
+----+---+
|Anna| 30|
+----+---+

--- 4. SUBTRACT (різниця DF1 - DF2) ---
+----+---+
|Name|Age|
+----+---+
|Ivan| 25|
+----+---+

--- 5. UNION BY NAME (безпечне об'єднання за назвами) ---
+------+---+
|  Name|Age|
+------+---+
|  Ivan| 25|
|  Anna| 30|
|  Ivan| 25|
|  Anna| 30|
|  Oleg| 35|
|Dmytro| 40|
+------+---+



### Оптимізація продуктивності з’єднань (Join Performance)

#### Фундамент продуктивності

- **Проєкція (Projection):** Операція відбору стовпців (`.select()`). Необхідна для обмеження об'єму даних, що потрапляють у пам'ять та передаються мережею.

- **Порядок операндів:** Незважаючи на алгоритми оптимізації Catalyst, рекомендується дотримуватися черговості операндів, де першим вказується менший за обсягом DataFrame.

- **Технічна логіка:** Це гарантує ефективне формування Build-таблиці (хеш-структури) в оперативній пам'яті вузлів, що мінімізує витрати ресурсів на ініціалізацію Join.

#### Broadcast Join

- **Проблема Shuffle:** Стандартні стратегії (*SortMergeJoin*) вимагають глобального перерозподілу даних між вузлами, що створює пікове навантаження на мережу.

- **Механізм Broadcast:** Копія Build-таблиці розсилається на всі Executor вузли.

- **Хеш-структура в RAM:** В пам'яті вузлів розміщуються ключі з'єднання та корисне навантаження (рядки після проєкції).

- **Результат:** Велика таблиця (Probe Table) обробляється локально. Це виключає фазу Shuffle та радикально прискорює запит.


Для перевірки стратегії виконання без доступу до Spark UI використовується метод `.explain()`.

```
from pyspark.sql.functions import broadcast

# 1. Застосування Проєкції (Projection) для мінімізації Build-таблиці
small_ready = df_small.select("key_id", "attr_name")

# 2. Формування Join з явною підказкою (Hint)
# Оптимізатор використовує small_ready як Build-таблицю для трансляції
result = df_large.join(broadcast(small_ready), "key_id", "inner")

# 3. Аналіз плану виконання (Execution Plan)
# Наявність вузла "BroadcastHashJoin" підтверджує успішну оптимізацію
result.explain()

```

**Важливо:** Якщо обсяг Build-таблиці перевищує ліміти RAM вузлів, Spark ініціює **Shuffle Spill** на диск або аварійно завершить процес (Out Of Memory).

## 12. Особливості роботи зі складними типами даних

У сучасній дата-інженерії, особливо при роботі з JSON-файлами або підготовці даних для ML-моделей (**feature engineering**), базових типів стає замало. Spark дозволяє зберігати дані ієрархічно, використовуючи три конструктори складних структур:

- **ArrayType:** впорядковані списки елементів.

- **StructType:** вкладені схеми з фіксованими іменами полів ("таблиця в рядку").

- **MapType:** гнучкі пари "ключ-значення" (аналог Python-словників).

Ці типи можна комбінувати, створюючи структури будь-якої глибини без розбиття даних на окремі таблиці.

### Обробка JSON: від рядків до структур

![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0320-databricks.png)

Часто дані надходять у форматі JSON-рядків. Проте робота з ними як із текстом є неефективною.

- **Проблема:** JSON-рядки вимагають постійного парсингу "на льоту", що витрачає пам'ять та ресурси процесора.

- **Рішення:** використання методу `from_json()` із заздалегідь визначеною схемою (`StructType`). Перетворення на нативні структури Spark забезпечує сувору перевірку типів та значно вищу продуктивність обробки.

![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0321-databricks.png)

Структури дозволяють логічно групувати пов'язані поля.

- **Доступ до полів:** Для звернення до вкладених даних використовується крапкова нотація (наприклад, `col("user.name")`) або спеціальний метод об'єкта Column – `getField()`.
- **Розгортання (Flattening):** Використання `select("user.*")` дозволяє швидко розгорнути всі поля структури в окремі стовпці верхнього рівня.

### Маніпуляції з масивами

![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0322-databricks.png)

При роботі з масивами часто виникає потреба розгорнути дані, зберігаючи при цьому значення в інших стовпцях. Для цього у Spark є функція `explode()`.

Наприклад, якщо ми маємо два рядки – ID 1 із масивом [A, B, C] та ID 2 із масивом [X, Y] – застосування `explode(items)` створить окремий рядок для кожного елемента масиву, зберігаючи зв’язок із відповідним ID.

Поширені функції для маніпуляцій зі стовпцями-масивами

| Функція | Призначення |
|----------|-------------|
| `array_contains(col, val)` | Перевіряє, чи містить масив вказане значення (`True`/`False`) |
| `size(col)` | Повертає кількість елементів у масиві |
| `element_at(col, n)` | Повертає `n`-й елемент масиву (індексація з 1) |
| `array_distinct(col)` | Видаляє дублікати зі стовпця-масиву |

### Агрегація в колекції

![image](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img0323-databricks.png)

Spark дозволяє згортати дані з декількох рядків у масиви в межах операції групування (`groupBy`), використовуючи функції збору:

- `collect_list(col)`: формує масив усіх значень стовпця. Зберігає дублікати та порядок елементів. Використовується для повної ретроспективи подій.

- `collect_set(col)`: формує множину унікальних значень. Видаляє дублікати, порядок елементів не гарантується. Використовується для визначення переліку унікальних категорій.

### Найкращі практики та продуктивність

Ефективна робота зі складними типами потребує суворого керування ресурсами кластера:

1. **Контроль "вибуху даних" (Data Explosion):** Обережно використовуйте `explode()`. Розгортання великих масивів створює мільйони рядків, що перевантажує CPU та дискову підсистему під час shuffle.

2. **Ризики OutOfMemory (OOM):** Функції `collect_list` та `collect_set` збирають дані в пам'ять одного вузла. Уникайте їх для груп із надвеликою кількістю записів.

3. **Ефективність зберігання:** Використовуйте `collect_set()`, якщо порядок елементів не важливий – це зменшує обсяг фінальних даних через видалення дублікатів.

### Приклад на роботу зі складними типами даних

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, explode, collect_set, size
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, DoubleType

spark = SparkSession.builder.appName("ComplexTypesMasterclass").getOrCreate()

# 1. Сирі дані (імітація логів з Kafka або JSON-файлів)
raw_data = [
    (101, '{"user": {"name": "Ivan", "city": "Kyiv"}, "items": [{"id": "A1", "price": 500.0}, {"id": "B2", "price": 1200.0}]}'),
    (102, '{"user": {"name": "Anna", "city": "Lviv"}, "items": [{"id": "A1", "price": 500.0}]}')
]

# 2. Визначення суворої схеми (Struct всередині Array)
item_schema = StructType([
    StructField("id", StringType()),
    StructField("price", DoubleType())
])

json_schema = StructType([
    StructField("user", StructType([
        StructField("name", StringType()),
        StructField("city", StringType())
    ])),
    StructField("items", ArrayType(item_schema))
])

# 3. Парсинг та обробка
df = spark.createDataFrame(raw_data, ["log_id", "payload"])

processed_df = df.withColumn("data", from_json(col("payload"), json_schema)) \
    .select(
        col("log_id"),
        col("data.user.name").alias("customer"),
        col("data.user.city").alias("city"),
        explode(col("data.items")).alias("item") # "Вибух" масиву товарів
    ) \
    .select("customer", "city", "item.id", "item.price")

processed_df.show()

# 4. Зворотна агрегація: отримання унікальних товарів за містами
final_report = processed_df.groupBy("city").agg(
    collect_set("id").alias("unique_items_sold"),
    size(collect_set("id")).alias("distinct_count")
)

final_report.show(truncate=False)

+--------+----+---+------+
|customer|city| id| price|
+--------+----+---+------+
|    Ivan|Kyiv| A1| 500.0|
|    Ivan|Kyiv| B2|1200.0|
|    Anna|Lviv| A1| 500.0|
+--------+----+---+------+

+----+-----------------+--------------+
|city|unique_items_sold|distinct_count|
+----+-----------------+--------------+
|Kyiv|[A1, B2]         |2             |
|Lviv|[A1]             |1             |
+----+-----------------+--------------+



## 13. Корисні ресурси

1. [Spark SQL, DataFrames and Datasets Guide](https://spark.apache.org/docs/latest/sql-programming-guide.html)

2. [PySpark API](https://spark.apache.org/docs/latest/api/python/)

3. [Оптимізація коду в PySpark з використанням найкращих практик](https://data-life-ua.com/coding/optymizatsiia-kodu-v-pyspark-z-vykorystanniam-naykrashchykh-praktyk/)
